# Urban Safety — Regression (Crime Score): Feature Inspection & Pre-ML Analysis

Loads pre-computed segment features with polynomial crime scores, explores the crime score target variable and feature distributions prior to regression modelling.

**See `03_London_RegressionML_CrimeScore.ipynb` for model training.**

## Cell 1 — Imports and config

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler

sns.set(rc={'figure.figsize': (8, 8)})
pd.set_option('display.float_format', '{:.3f}'.format)

FEATURE_COLS = ['lighting', 'visibility', 'connectivity', 'enclosure']

## Cell 2 — Load pre-computed features
Loads `csvFiles/segment_polynomial_crime-scores_w-id.csv`. Computes `crime_score_scaled` if not already present.

In [ ]:
CRIME_FEATURES_CSV = 'csvFiles/segment_polynomial_crime-scores_w-id.csv'
print(f'Loading: {CRIME_FEATURES_CSV}')
features = pd.read_csv(CRIME_FEATURES_CSV)
print(f'DEBUG: loaded columns -> {list(features.columns)}')

required_cols = FEATURE_COLS + ['location_id', 'borough']
missing = [c for c in required_cols if c not in features.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

if 'crime_score_scaled' not in features.columns:
    if 'crime_score' in features.columns:
        from sklearn.preprocessing import StandardScaler as _SS
        _sc = _SS()
        features['crime_score_scaled'] = _sc.fit_transform(features[['crime_score']])
        print('Computed crime_score_scaled from crime_score.')
    else:
        raise ValueError("CSV must contain 'crime_score_scaled' or 'crime_score'.")

print(f'\u2713 Loaded {len(features):,} segments with {len(features.columns)} columns')
display(features.head())
print(f'\nCrime count statistics:')
print(features['crime_count'].describe().round(3))
print(f'\nFeature statistics:')
display(features[FEATURE_COLS].describe().round(3))

## Cell 3 — Crime Score Distribution Check

In [ ]:
print('=== CRIME SCORE DISTRIBUTION ===\n')
crime_scores = features['crime_score'].values
print(f'Total segments: {len(crime_scores):,}')
for label, val in [('Min', crime_scores.min()), ('Q1', np.percentile(crime_scores, 25)),
                   ('Median', np.percentile(crime_scores, 50)), ('Mean', crime_scores.mean()),
                   ('Q3', np.percentile(crime_scores, 75)), ('Max', crime_scores.max()),
                   ('Std Dev', crime_scores.std())]:
    print(f'  {label:8s}: {val:.2f}')
zero = (crime_scores == 0).sum()
print(f'\n  Zero score: {zero:,} ({100*zero/len(crime_scores):.1f}%)  '
      f'Non-zero: {len(crime_scores)-zero:,} ({100*(len(crime_scores)-zero)/len(crime_scores):.1f}%)')

print(f'\nPer-borough mean crime score:')
bm = features.groupby('borough')['crime_score'].agg(['mean','count']).sort_values('mean', ascending=False)
bm.index = [b.replace('London Borough of ','').replace(', London, UK','').replace('City of ','') for b in bm.index]
print(bm.round(2).to_string())

In [ ]:
os.makedirs('plots', exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(features['crime_score'], bins=80, color='steelblue', alpha=0.7, edgecolor='none')
axes[0].set_xlabel('Crime Score'); axes[0].set_ylabel('Number of Segments')
axes[0].set_title('Distribution of Crime Scores Across All Segments')
axes[0].spines[['top','right']].set_visible(False); axes[0].grid(True, alpha=0.3, axis='y', linestyle='--')

nonzero = features[features['crime_score'] > 0]['crime_score']
axes[1].hist(nonzero, bins=80, color='coral', alpha=0.7, edgecolor='none')
axes[1].set_xlabel('Crime Score'); axes[1].set_ylabel('Frequency (log scale)')
axes[1].set_yscale('log')
axes[1].set_title(f'Distribution of Crime Scores (Non-Zero Only, n={len(nonzero):,})')
axes[1].spines[['top','right']].set_visible(False); axes[1].grid(True, alpha=0.3, axis='y', linestyle='--')

plt.tight_layout()
plt.savefig('plots/crime_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
fc = features.copy()
fc['borough_short'] = (fc['borough'].str.replace('London Borough of ','')
    .str.replace(', London, UK','').str.replace('City of ',''))
fc.boxplot(column='crime_score', by='borough_short', ax=ax)
ax.set_xlabel('Borough'); ax.set_ylabel('Crime Score')
ax.set_title('Crime Score Distribution by Borough')
plt.suptitle(''); plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/crime_score_by_borough.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature EDA — inspect the data and how it correlates

In [ ]:
feat_df = features[FEATURE_COLS].copy()
print(f'Feature dataframe: {feat_df.shape[0]:,} segments x {feat_df.shape[1]} features')
display(feat_df.head(10))
summary = feat_df.describe().T
summary['skew'] = feat_df.skew()
print('Summary (with skew):')
display(summary.round(3))

import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style='whitegrid')
plt.figure(figsize=(6, 5))
corr = features[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature correlation matrix (Pearson)')
plt.tight_layout()
os.makedirs('plots', exist_ok=True)
plt.savefig('plots/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print(corr.round(3))

In [ ]:
from pandas.plotting import parallel_coordinates
from matplotlib.patches import Patch

_z = (features[FEATURE_COLS] - features[FEATURE_COLS].mean()) / features[FEATURE_COLS].std()
_z['crime_score'] = features['crime_score'].values
_z = _z.dropna(subset=['crime_score']).sample(n=min(800, len(_z)), random_state=42)

all_nonzero = features[features['crime_score'] > 0]['crime_score']
q25 = all_nonzero.quantile(0.25); q50 = all_nonzero.quantile(0.50)
q75 = all_nonzero.quantile(0.75); max_v = features['crime_score'].max()

_color_map = {'Very Low': '#808080', 'Low': '#2ca02c', 'Medium': '#ffd700',
               'High': '#ff7f0e', 'Very High': '#d62728'}

fig, ax = plt.subplots(figsize=(11, 6))
_z['crime_score_bin'] = pd.cut(_z['crime_score'],
    bins=[-0.1, 0.0001, q25, q50, q75, max_v + 1],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'], include_lowest=True)
_present = sorted(_z['crime_score_bin'].unique().tolist())
parallel_coordinates(_z[FEATURE_COLS + ['crime_score_bin']], 'crime_score_bin',
                     color=[_color_map[c] for c in _present], alpha=0.5, ax=ax)
ax.legend(handles=[Patch(facecolor=_color_map[c], label=c) for c in _present])
ax.set_title('Parallel coordinates by crime score (z-scored features, sampled)')
ax.set_ylabel('Feature value (z-scored)')
_y_min = min(_z[FEATURE_COLS].min().min(), -3); _y_max = max(_z[FEATURE_COLS].max().max(), 3)
ax.set_ylim(_y_min - 0.3, _y_max + 0.3)
plt.tight_layout()
plt.savefig('plots/parallel_coordinates_crimescore.png', dpi=150, bbox_inches='tight')
plt.show()

# Excluding zero-score segments
_znz = (features[FEATURE_COLS] - features[FEATURE_COLS].mean()) / features[FEATURE_COLS].std()
_znz['crime_score'] = features['crime_score'].values
_znz = _znz[_znz['crime_score'] > 0].sample(n=min(800, len(_znz)), random_state=42)
_znz['crime_score_bin'] = pd.cut(_znz['crime_score'],
    bins=[-0.1, 0.0001, q25, q50, q75, max_v + 1],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'], include_lowest=True)
_znz_f = _znz[_znz['crime_score_bin'] != 'Very Low']
_present2 = sorted(_znz_f['crime_score_bin'].unique().tolist())
fig, ax = plt.subplots(figsize=(11, 6))
parallel_coordinates(_znz_f[FEATURE_COLS + ['crime_score_bin']], 'crime_score_bin',
                     color=[_color_map[c] for c in _present2], alpha=0.5, ax=ax)
ax.legend(handles=[Patch(facecolor=_color_map[c], label=c) for c in _present2])
ax.set_title('Parallel coordinates by crime score - excluding zero-score segments (sampled)')
ax.set_ylabel('Feature value (z-scored)')
ax.set_ylim(_y_min - 0.3, _y_max + 0.3)
plt.tight_layout()
plt.savefig('plots/parallel_coordinates_crimescore_nonzero.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Non-zero segments shown: {len(_znz_f):,}')

## Target Variable: Crime Score

In [ ]:
print('=== Crime Score Target Variable ===\n')
cs = features['crime_score']
print(f'Total segments: {len(features):,}')
print(f'Segments with crimes: {(cs > 0).sum():,}')
print(f'Segments with no crimes: {(cs == 0).sum():,}')
print(f'\nCrime score statistics:')
for label, val in [('Min', cs.min()), ('Max', cs.max()), ('Mean', cs.mean()),
                   ('Median', cs.median()), ('Std Dev', cs.std()),
                   ('Q1 (25%)', cs.quantile(0.25)), ('Q3 (75%)', cs.quantile(0.75))]:
    print(f'  {label:12s}: {val:.4f}')
print(f'\nCrime score percentiles:')
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'  P{p:2d}: {cs.quantile(p/100):8.4f}')

## Standard scale the crime score

In [ ]:
crime_scores_arr = features['crime_score'].values
scaler_cs = StandardScaler()
scaled = scaler_cs.fit_transform(crime_scores_arr.reshape(-1, 1)).flatten()

print(f'Original crime scores - Min: {crime_scores_arr.min():.4f}, Max: {crime_scores_arr.max():.4f}, '
      f'Mean: {crime_scores_arr.mean():.4f}, Std: {crime_scores_arr.std():.4f}')
print(f'Scaled crime scores   - Min: {scaled.min():.4f}, Max: {scaled.max():.4f}, '
      f'Mean: {scaled.mean():.6f}, Std: {scaled.std():.4f}')
print(f'Total segments: {len(scaled)}')